In [7]:
import json
import logging
import re
import sys
from typing import Dict, Any

# 🪵 Colab Logging Setup (Prevents duplicate outputs in Colab cells)
logger = logging.getLogger()
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))
    logger.addHandler(handler)

In [13]:
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely."""
    try:
        # Sanitize input: allow only digits, operators, dots, and brackets
        sanitized = re.sub(r'[^0-9+\-*/().\s]', '', expression)
        if not sanitized.strip():
            return "Error in calculation"
        return str(eval(sanitized))
    except Exception:
        return "Error in calculation"

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        # Strip simple punctuation symbols for clean word tokenization
        clean_text = re.sub(r'[^\w\s]', '', text)
        words = clean_text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

def text_stats(text: str) -> dict:
    """🚀 BONUS TOOL: Evaluates basic string metrics."""
    try:
        words = text.split()
        return {
            "word_count": len(words),
            "char_count": len(text),
            "uppercase_count": sum(1 for w in words if w.isupper() and len(w) > 1)
        }
    except Exception:
        return {}

In [16]:
from typing import Dict, Any, Callable
import logging
import re

logger = logging.getLogger()

# Define tool configurations globally to avoid recreating them on each agent call
_TOOLS_CONFIG = [
    {
        "function_name": "calculator",
        "description": "Calculates the result of a mathematical expression.",
        "keywords": ["calculate", "math", "add", "subtract", "multiply", "divide"]
    },
    {
        "function_name": "extract_keywords",
        "description": "Extracts key terms from a given text.",
        "keywords": ["keywords", "extract", "terms"]
    },
    {
        "function_name": "text_stats",
        "description": "Provides basic statistics about a given text, such as word count and character count.",
        "keywords": ["analyze", "statistics", "stats", "text", "string"]
    }
]

# Map function names to actual function objects for direct dispatch
_TOOL_FUNCTIONS: Dict[str, Callable] = {
    "calculator": calculator,
    "extract_keywords": extract_keywords,
    "text_stats": text_stats
}

def agent(query: str) -> Dict[str, Any]:
    """Simulates an agent that routes queries to appropriate tools."""
    query_lower = query.lower() # Convert query to lowercase once
    selected_tool_config = None

    for tool_config in _TOOLS_CONFIG:
        if any(keyword in query_lower for keyword in tool_config["keywords"]):
            selected_tool_config = tool_config
            break

    if selected_tool_config:
        tool_name = selected_tool_config["function_name"]
        logger.info(f"Agent selected tool: {tool_name}")

        tool_func = _TOOL_FUNCTIONS.get(tool_name)
        if not tool_func:
            logger.error(f"Function {tool_name} not found in _TOOL_FUNCTIONS mapping.")
            return {"tool": "error", "response": "Internal tool mapping error."}

        # Dynamic argument extraction based on tool_name
        if tool_name == "calculator":
            result = tool_func(query)
        elif tool_name == "extract_keywords":
            match = re.search(r'from\s*(.*)', query, re.IGNORECASE)
            text_to_extract = match.group(1) if match else query
            result = tool_func(text_to_extract)
        elif tool_name == "text_stats":
            match = re.search(r'(?:this|string)\s*(.*)', query, re.IGNORECASE)
            text_to_analyze = match.group(1) if match else query
            result = tool_func(text_to_analyze)
        else:
            # This case should ideally not be reached if _TOOLS_CONFIG and _TOOL_FUNCTIONS are in sync
            logger.warning(f"Unknown tool_name encountered: {tool_name}")
            return {"tool": "none", "response": "I couldn't find a specific tool to handle your request, but I can't answer general questions."}

        return {"tool": tool_name, "result": result}
    else:
        logger.info("Agent found no suitable tool.")
        return {"tool": "none", "response": "I couldn't find a specific tool to handle your request, but I can't answer general questions."}


In [17]:
print("="*60)
print("🚀 RUNNING AGENT PIPELINE SIMULATION")
print("="*60)

test_queries = [
    "Calculate 20 + 5 * 2",
    "Extract keywords from Artificial Intelligence is transforming engineering industries",
    "Analyze this specific string configuration right now.",
    "What is machine learning?"
]

for q in test_queries:
    print(f"\nQuery: {q}")
    response = agent(q)
    print(f"Structured Response:\n{json.dumps(response, indent=2)}")
    print("-" * 50)

INFO:root:Agent selected tool: calculator
INFO:root:Agent selected tool: extract_keywords
INFO:root:Agent selected tool: text_stats
INFO:root:Agent found no suitable tool.


🚀 RUNNING AGENT PIPELINE SIMULATION

Query: Calculate 20 + 5 * 2
Structured Response:
{
  "tool": "calculator",
  "result": "30"
}
--------------------------------------------------

Query: Extract keywords from Artificial Intelligence is transforming engineering industries
Structured Response:
{
  "tool": "extract_keywords",
  "result": [
    "industries",
    "intelligence",
    "artificial",
    "engineering",
    "transforming"
  ]
}
--------------------------------------------------

Query: Analyze this specific string configuration right now.
Structured Response:
{
  "tool": "text_stats",
  "result": {
    "word_count": 5,
    "char_count": 40,
    "uppercase_count": 0
  }
}
--------------------------------------------------

Query: What is machine learning?
Structured Response:
{
  "tool": "none",
  "response": "I couldn't find a specific tool to handle your request, but I can't answer general questions."
}
--------------------------------------------------


If you want to run the `agent` with a specific query without entering the continuous interactive playground, you can call the `agent` function directly:

In [15]:

single_query = "Calculate 10 + 3 * 7"
single_response = agent(single_query)
print(f"Single Query: {single_query}")
print(f"Response:\n{json.dumps(single_response, indent=2)}")

single_query_2 = "Extract keywords from the quick brown fox jumps over the lazy dog"
single_response_2 = agent(single_query_2)
print(f"\nSingle Query: {single_query_2}")
print(f"Response:\n{json.dumps(single_response_2, indent=2)}")

INFO:root:Agent selected tool: calculator
INFO:root:Agent selected tool: extract_keywords


Single Query: Calculate 10 + 3 * 7
Response:
{
  "tool": "calculator",
  "result": "31"
}

Single Query: Extract keywords from the quick brown fox jumps over the lazy dog
Response:
{
  "tool": "extract_keywords",
  "result": [
    "jumps",
    "brown",
    "quick"
  ]
}
